# 05C — Incident-Held-Out Probabilistic Baseline Modeling

**Build ID: `05C_V1_INCIDENT_HELD_OUT_BASELINE_MODELING`**

05B froze:

- six scientific feature-set ablations;
- five outer incident-held-out folds;
- nested inner grouped folds;
- three training-weight regimes.

05C performs the **first learnability experiment**.

## Important design choice

No hyperparameter tuning is performed in 05C.

A fixed LightGBM configuration is declared before seeing outer-test
performance. This gives a relatively clean first answer to:

> Is dynamic wildfire fireline hold/burn-over behavior learnable on completely
> unseen wildfire incidents, and which information sources carry that skill?

The nested inner folds remain frozen for later tuning/calibration work.

## Experiments

### Feature-source ablation — incident-equal training

1. `A_APPROACH_ONLY`
2. `B_STATIC_ENV_CORE`
3. `C_NEAR_WEATHER_ONLY`
4. `D_FULL_CORE`
5. `E_FULL_EXTENDED`
6. `F_FULL_COVERAGE_STABLE`

### Weighting ablation — full extended feature set

7. `E_FULL_EXTENDED` + ordinary row weighting
8. `E_FULL_EXTENDED` + incident-class-equal weighting

`E_FULL_EXTENDED + incident_equal` is already included in the feature ablation,
so the full weighting comparison contains all three regimes without duplicate
training.

A training-prevalence prior is also evaluated as a no-skill probabilistic
reference.

## Evaluation contract

Every predicted probability in the final OOF table comes from a model that
**never trained on that wildfire incident**.

Training weights affect model fitting only. Test metrics are not reweighted.

For each experiment:

- pooled PR-AUC;
- pooled ROC-AUC;
- Brier score;
- log loss;
- incident-macro Brier and log loss;
- mixed-incident macro PR-AUC and ROC-AUC;
- calibration intercept and slope;
- reliability bins and descriptive ECE.

No classification threshold is selected in 05C.

## 0. Configuration

In [ ]:
import os  # Configurable project root for the public release.
NOTEBOOK_BUILD_ID = (
    "05C_V1_INCIDENT_HELD_OUT_BASELINE_MODELING"
)

from pathlib import Path
import sys
import subprocess
import importlib.util

PROJECT_ROOT = Path(
    os.environ.get("FIRELINE_PROJECT_ROOT", ".")
)

MASTER_PATH = (
    PROJECT_ROOT / "data" / "processed" /
    "modeling_table_assembly_v1" /
    "fireline_hold_modeling_master_v1.parquet"
)

UNIT_DESIGN_PATH = (
    PROJECT_ROOT / "data" / "processed" /
    "modeling_design_v1" /
    "unit_outer_fold_and_training_weights_v1.parquet"
)

FEATURE_SETS_PATH = (
    PROJECT_ROOT / "outputs" /
    "modeling_design_v1" /
    "FEATURE_SETS_V1.json"
)

OUTER_INCIDENT_FOLDS_PATH = (
    PROJECT_ROOT / "outputs" /
    "modeling_design_v1" /
    "OUTER_INCIDENT_FOLDS_V1.csv"
)

EVALUATION_PROTOCOL_PATH = (
    PROJECT_ROOT / "outputs" /
    "modeling_design_v1" /
    "EVALUATION_PROTOCOL_V1.json"
)

OUTPUT_ROOT = (
    PROJECT_ROOT / "outputs" /
    "baseline_modeling_v1"
)

PREDICTION_ROOT = (
    PROJECT_ROOT / "data" / "processed" /
    "baseline_modeling_v1" /
    "oof_predictions"
)

for p in [
    OUTPUT_ROOT,
    PREDICTION_ROOT,
]:
    p.mkdir(
        parents=True,
        exist_ok=True,
    )

for p in [
    MASTER_PATH,
    UNIT_DESIGN_PATH,
    FEATURE_SETS_PATH,
    OUTER_INCIDENT_FOLDS_PATH,
    EVALUATION_PROTOCOL_PATH,
]:
    assert p.exists(), p

EXPECTED_UNITS = 304_143
EXPECTED_INCIDENTS = 381
OUTER_FOLDS = 5

MODEL_FAMILY = "LightGBM"

# Fixed a-priori baseline parameters.
# No outer-test-based tuning is allowed in 05C.
LGBM_PARAMS = {
    "objective": "binary",
    "n_estimators": 300,
    "learning_rate": 0.05,
    "num_leaves": 31,
    "max_depth": -1,
    "min_child_samples": 150,
    "subsample": 0.85,
    "colsample_bytree": 0.80,
    "reg_alpha": 0.0,
    "reg_lambda": 1.0,
    "max_bin": 127,
    "n_jobs": -1,
    "verbosity": -1,
}

MODEL_SEED = 20260825

PRED_EPS = 1e-6
CALIBRATION_BINS = 10

print("=" * 100)
print("RUNNING:", NOTEBOOK_BUILD_ID)
print("=" * 100)
print("Model family:", MODEL_FAMILY)
print("Fixed parameters:", LGBM_PARAMS)

## 1. Imports and package versions

In [ ]:
required = {
    "pandas": "pandas",
    "numpy": "numpy",
    "pyarrow": "pyarrow",
    "sklearn": "scikit-learn",
    "scipy": "scipy",
    "lightgbm": "lightgbm",
}

for module, package in required.items():
    if importlib.util.find_spec(
        module
    ) is None:
        subprocess.check_call(
            [
                sys.executable,
                "-m",
                "pip",
                "install",
                package,
            ]
        )

import gc
import json
import math
import time
import warnings

from datetime import datetime, timezone

import numpy as np
import pandas as pd
import pyarrow.parquet as pq

import sklearn
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    brier_score_loss,
    log_loss,
)
from scipy.optimize import minimize

import lightgbm as lgb
from lightgbm import LGBMClassifier

pd.set_option(
    "display.max_columns",
    500,
)
pd.set_option(
    "display.max_rows",
    500,
)
pd.set_option(
    "display.width",
    420,
)

versions = pd.DataFrame(
    [
        ["python", sys.version.split()[0]],
        ["numpy", np.__version__],
        ["pandas", pd.__version__],
        ["scikit-learn", sklearn.__version__],
        ["lightgbm", lgb.__version__],
    ],
    columns=[
        "package",
        "version",
    ],
)

display(
    versions
)

versions.to_csv(
    OUTPUT_ROOT /
    "SOFTWARE_VERSIONS_V1.csv",
    index=False,
)

## 2. Load frozen 05B design

In [ ]:
with FEATURE_SETS_PATH.open(
    "r",
    encoding="utf-8",
) as f:
    FEATURE_SETS = json.load(
        f
    )

with EVALUATION_PROTOCOL_PATH.open(
    "r",
    encoding="utf-8",
) as f:
    EVALUATION_PROTOCOL = json.load(
        f
    )

unit_design = pd.read_parquet(
    UNIT_DESIGN_PATH
)

outer_incidents = pd.read_csv(
    OUTER_INCIDENT_FOLDS_PATH
)

assert len(
    unit_design
) == EXPECTED_UNITS

assert unit_design[
    "unit_uid"
].is_unique

assert unit_design[
    "_irwin_norm"
].nunique() == EXPECTED_INCIDENTS

assert set(
    unit_design[
        "outer_fold"
    ]
) == set(
    range(
        OUTER_FOLDS
    )
)

assert (
    unit_design.groupby(
        "_irwin_norm"
    )[
        "outer_fold"
    ].nunique()
    == 1
).all()

assert set(
    FEATURE_SETS
) == {
    "A_APPROACH_ONLY",
    "B_STATIC_ENV_CORE",
    "C_NEAR_WEATHER_ONLY",
    "D_FULL_CORE",
    "E_FULL_EXTENDED",
    "F_FULL_COVERAGE_STABLE",
}

for set_name, cols in FEATURE_SETS.items():

    assert len(
        cols
    ) > 0

    assert len(
        cols
    ) == len(
        set(
            cols
        )
    )

master_schema = set(
    pq.read_schema(
        MASTER_PATH
    ).names
)

for set_name, cols in FEATURE_SETS.items():

    missing = sorted(
        set(
            cols
        )
        - master_schema
    )

    if missing:
        raise RuntimeError(
            f"{set_name} missing master columns: {missing[:20]}"
        )

display(
    pd.DataFrame(
        [
            {
                "feature_set": k,
                "columns": len(
                    v
                ),
            }
            for k, v in FEATURE_SETS.items()
        ]
    )
)

print(
    "05B DESIGN CONTRACT: PASS"
)

## 3. Freeze baseline experiment matrix

The feature-ablation comparison uses **incident-equal training** throughout so
the feature-source conclusion is not confounded by changing the loss weighting.

The weighting ablation then holds the largest full feature set fixed.

In [ ]:
EXPERIMENTS = [
    {
        "experiment_id":
        "A_APPROACH_ONLY__incident_equal",
        "feature_set":
        "A_APPROACH_ONLY",
        "weight_scheme":
        "incident_equal",
        "experiment_group":
        "feature_ablation",
    },
    {
        "experiment_id":
        "B_STATIC_ENV_CORE__incident_equal",
        "feature_set":
        "B_STATIC_ENV_CORE",
        "weight_scheme":
        "incident_equal",
        "experiment_group":
        "feature_ablation",
    },
    {
        "experiment_id":
        "C_NEAR_WEATHER_ONLY__incident_equal",
        "feature_set":
        "C_NEAR_WEATHER_ONLY",
        "weight_scheme":
        "incident_equal",
        "experiment_group":
        "feature_ablation",
    },
    {
        "experiment_id":
        "D_FULL_CORE__incident_equal",
        "feature_set":
        "D_FULL_CORE",
        "weight_scheme":
        "incident_equal",
        "experiment_group":
        "feature_ablation",
    },
    {
        "experiment_id":
        "E_FULL_EXTENDED__incident_equal",
        "feature_set":
        "E_FULL_EXTENDED",
        "weight_scheme":
        "incident_equal",
        "experiment_group":
        "feature_and_weight_ablation",
    },
    {
        "experiment_id":
        "F_FULL_COVERAGE_STABLE__incident_equal",
        "feature_set":
        "F_FULL_COVERAGE_STABLE",
        "weight_scheme":
        "incident_equal",
        "experiment_group":
        "feature_ablation",
    },
    {
        "experiment_id":
        "E_FULL_EXTENDED__row",
        "feature_set":
        "E_FULL_EXTENDED",
        "weight_scheme":
        "row",
        "experiment_group":
        "weight_ablation",
    },
    {
        "experiment_id":
        "E_FULL_EXTENDED__incident_class_equal",
        "feature_set":
        "E_FULL_EXTENDED",
        "weight_scheme":
        "incident_class_equal",
        "experiment_group":
        "weight_ablation",
    },
]

experiment_table = pd.DataFrame(
    EXPERIMENTS
)

experiment_table[
    "feature_columns"
] = experiment_table[
    "feature_set"
].map(
    lambda x: len(
        FEATURE_SETS[
            x
        ]
    )
)

assert experiment_table[
    "experiment_id"
].is_unique

display(
    experiment_table
)

experiment_table.to_csv(
    OUTPUT_ROOT /
    "BASELINE_EXPERIMENT_MATRIX_V1.csv",
    index=False,
)

## 4. Metric helpers

Incident-macro Brier/log-loss include all incidents.

Incident-macro AUC metrics include only incidents containing both target
classes because AUC is undefined for single-class incidents.

In [ ]:
def clip_prob(
    p,
):
    return np.clip(
        np.asarray(
            p,
            dtype=float,
        ),
        PRED_EPS,
        1.0
        - PRED_EPS,
    )


def binary_log_loss_vector(
    y,
    p,
):
    y = np.asarray(
        y,
        dtype=float,
    )

    p = clip_prob(
        p
    )

    return -(
        y
        * np.log(
            p
        )
        + (
            1.0
            - y
        )
        * np.log(
            1.0
            - p
        )
    )


def calibration_intercept_slope(
    y,
    p,
):
    y = np.asarray(
        y,
        dtype=float,
    )

    p = clip_prob(
        p
    )

    z = np.log(
        p
        / (
            1.0
            - p
        )
    )

    def nll(
        theta,
    ):
        intercept = theta[
            0
        ]
        slope = theta[
            1
        ]

        eta = (
            intercept
            + slope
            * z
        )

        # Stable logistic loss:
        return float(
            np.sum(
                np.logaddexp(
                    0.0,
                    eta,
                )
                - y
                * eta
            )
        )

    result = minimize(
        nll,
        x0=np.array(
            [
                0.0,
                1.0,
            ]
        ),
        method="BFGS",
    )

    if not result.success:

        warnings.warn(
            "Calibration intercept/slope optimizer did not fully converge: "
            + str(
                result.message
            )
        )

    return (
        float(
            result.x[
                0
            ]
        ),
        float(
            result.x[
                1
            ]
        ),
    )


def incident_metrics(
    frame,
):
    rows = []

    for incident, g in frame.groupby(
        "_irwin_norm",
        sort=False,
    ):

        y = g[
            "hold_label"
        ].to_numpy(
            int
        )

        p = g[
            "pred_prob"
        ].to_numpy(
            float
        )

        mixed = (
            np.unique(
                y
            ).size
            == 2
        )

        rows.append({
            "_irwin_norm":
            incident,

            "units":
            len(
                g
            ),

            "held_rate":
            float(
                np.mean(
                    y
                )
            ),

            "brier":
            float(
                np.mean(
                    (
                        p
                        - y
                    ) ** 2
                )
            ),

            "log_loss":
            float(
                np.mean(
                    binary_log_loss_vector(
                        y,
                        p,
                    )
                )
            ),

            "mixed_class":
            mixed,

            "pr_auc":
            (
                float(
                    average_precision_score(
                        y,
                        p,
                    )
                )
                if mixed
                else np.nan
            ),

            "roc_auc":
            (
                float(
                    roc_auc_score(
                        y,
                        p,
                    )
                )
                if mixed
                else np.nan
            ),
        })

    return pd.DataFrame(
        rows
    )


def reliability_table(
    frame,
    bins=10,
):
    x = frame[
        [
            "hold_label",
            "pred_prob",
        ]
    ].copy()

    try:

        x[
            "bin"
        ] = pd.qcut(
            x[
                "pred_prob"
            ],
            q=bins,
            duplicates="drop",
        )

    except ValueError:

        x[
            "bin"
        ] = pd.cut(
            x[
                "pred_prob"
            ],
            bins=bins,
            include_lowest=True,
        )

    rel = (
        x.groupby(
            "bin",
            observed=True,
            as_index=False,
        )
        .agg(
            units=(
                "hold_label",
                "size",
            ),
            mean_pred=(
                "pred_prob",
                "mean",
            ),
            observed_rate=(
                "hold_label",
                "mean",
            ),
        )
    )

    rel[
        "abs_calibration_gap"
    ] = (
        rel[
            "mean_pred"
        ]
        - rel[
            "observed_rate"
        ]
    ).abs()

    rel[
        "weight"
    ] = (
        rel[
            "units"
        ]
        / rel[
            "units"
        ].sum()
    )

    ece = float(
        (
            rel[
                "weight"
            ]
            * rel[
                "abs_calibration_gap"
            ]
        ).sum()
    )

    return (
        rel,
        ece,
    )


def summarize_predictions(
    frame,
):
    y = frame[
        "hold_label"
    ].to_numpy(
        int
    )

    p = frame[
        "pred_prob"
    ].to_numpy(
        float
    )

    assert np.unique(
        y
    ).size == 2

    inc = incident_metrics(
        frame
    )

    mixed = inc[
        "mixed_class"
    ]

    intercept, slope = (
        calibration_intercept_slope(
            y,
            p,
        )
    )

    rel, ece = reliability_table(
        frame,
        bins=CALIBRATION_BINS,
    )

    metrics = {
        "units":
        len(
            frame
        ),

        "incidents":
        frame[
            "_irwin_norm"
        ].nunique(),

        "held_rate":
        float(
            np.mean(
                y
            )
        ),

        "mean_pred":
        float(
            np.mean(
                p
            )
        ),

        "pooled_pr_auc":
        float(
            average_precision_score(
                y,
                p,
            )
        ),

        "pooled_roc_auc":
        float(
            roc_auc_score(
                y,
                p,
            )
        ),

        "pooled_brier":
        float(
            brier_score_loss(
                y,
                p,
            )
        ),

        "pooled_log_loss":
        float(
            log_loss(
                y,
                clip_prob(
                    p
                ),
                labels=[
                    0,
                    1,
                ],
            )
        ),

        "incident_macro_brier":
        float(
            inc[
                "brier"
            ].mean()
        ),

        "incident_macro_log_loss":
        float(
            inc[
                "log_loss"
            ].mean()
        ),

        "mixed_incidents":
        int(
            mixed.sum()
        ),

        "mixed_incident_macro_pr_auc":
        float(
            inc.loc[
                mixed,
                "pr_auc",
            ].mean()
        ),

        "mixed_incident_macro_roc_auc":
        float(
            inc.loc[
                mixed,
                "roc_auc",
            ].mean()
        ),

        "calibration_intercept":
        intercept,

        "calibration_slope":
        slope,

        "ece":
        ece,
    }

    return (
        metrics,
        inc,
        rel,
    )

## 5. No-skill training-prevalence prior

For each outer fold, the baseline probability equals the Held prevalence in the
other four folds.

This baseline is also incident-held-out because the test incident contributes
nothing to its probability estimate.

In [ ]:
prior_frames = []

for outer_fold in range(
    OUTER_FOLDS
):

    train = unit_design[
        unit_design[
            "outer_fold"
        ] != outer_fold
    ]

    test = unit_design[
        unit_design[
            "outer_fold"
        ] == outer_fold
    ].copy()

    prior = float(
        train[
            "hold_label"
        ].mean()
    )

    test[
        "pred_prob"
    ] = prior

    test[
        "experiment_id"
    ] = "TRAINING_PREVALENCE_PRIOR"

    test[
        "feature_set"
    ] = "NONE"

    test[
        "weight_scheme"
    ] = "row"

    prior_frames.append(
        test[
            [
                "unit_uid",
                "_irwin_norm",
                "IncidentName",
                "Year",
                "hold_label",
                "outer_fold",
                "pred_prob",
                "experiment_id",
                "feature_set",
                "weight_scheme",
            ]
        ]
    )


prior_oof = pd.concat(
    prior_frames,
    ignore_index=True,
)

assert len(
    prior_oof
) == EXPECTED_UNITS

assert prior_oof[
    "unit_uid"
].is_unique

prior_path = (
    PREDICTION_ROOT /
    "TRAINING_PREVALENCE_PRIOR.parquet"
)

prior_oof.to_parquet(
    prior_path,
    index=False,
)

prior_metrics, prior_incident, prior_rel = (
    summarize_predictions(
        prior_oof
    )
)

display(
    pd.DataFrame(
        [
            prior_metrics
        ]
    )
)

print(
    "NO-SKILL PRIOR: COMPLETE"
)

## 6. Training helpers

### Missing values

LightGBM handles missing numeric values natively. No global imputation is
performed.

### Weight normalization

Each raw training-weight vector is multiplied by a single fold-specific
constant so its mean equals 1.

This preserves all relative weights while keeping the overall objective scale
comparable across weighting schemes, which matters because tree regularization
is sensitive to the absolute scale of sample weights.

In [ ]:
WEIGHT_COLUMN = {
    "row":
    "weight_row_raw",

    "incident_equal":
    "weight_incident_equal_raw",

    "incident_class_equal":
    "weight_incident_class_equal_raw",
}


def normalized_training_weights(
    design_train,
    scheme,
):
    column = WEIGHT_COLUMN[
        scheme
    ]

    raw = design_train[
        column
    ].to_numpy(
        dtype=float
    )

    assert np.isfinite(
        raw
    ).all()

    assert (
        raw > 0
    ).all()

    scale = (
        len(
            raw
        )
        / raw.sum()
    )

    weights = (
        raw
        * scale
    )

    assert np.isclose(
        weights.mean(),
        1.0,
        rtol=0,
        atol=1e-10,
    )

    return weights


def make_model(
    seed,
):
    params = dict(
        LGBM_PARAMS
    )

    params[
        "random_state"
    ] = int(
        seed
    )

    return LGBMClassifier(
        **params
    )


def feature_frame_to_float32(
    frame,
    features,
):
    # All 05B features were frozen as numeric/bool.
    # Convert to a compact numeric frame while preserving NaN.
    out = pd.DataFrame(
        index=frame.index
    )

    for c in features:

        out[
            c
        ] = pd.to_numeric(
            frame[
                c
            ],
            errors="coerce",
        ).astype(
            np.float32
        )

    return out

## 7. Run all LightGBM outer-fold experiments

The notebook groups experiments by feature set, so `E_FULL_EXTENDED` is read
from Parquet once and reused for its three weighting schemes.

Each output prediction shard contains all 304,143 units exactly once.

In [ ]:
all_fold_metric_rows = []
all_runtime_rows = []

prediction_paths = {}

experiments_by_feature_set = {}

for experiment in EXPERIMENTS:

    experiments_by_feature_set.setdefault(
        experiment[
            "feature_set"
        ],
        [],
    ).append(
        experiment
    )


for feature_set, experiments_here in (
    experiments_by_feature_set.items()
):

    features = FEATURE_SETS[
        feature_set
    ]

    print()
    print("=" * 110)
    print(
        "FEATURE SET:",
        feature_set,
        "| columns:",
        len(
            features
        ),
        "| experiments:",
        len(
            experiments_here
        ),
    )
    print("=" * 110)

    load_columns = [
        "unit_uid",
        *features,
    ]

    feature_table = pd.read_parquet(
        MASTER_PATH,
        columns=load_columns,
    )

    assert len(
        feature_table
    ) == EXPECTED_UNITS

    assert feature_table[
        "unit_uid"
    ].is_unique

    # Align to the frozen unit design.
    data = unit_design[
        [
            "unit_uid",
            "_irwin_norm",
            "IncidentName",
            "Year",
            "hold_label",
            "outer_fold",
            "weight_row_raw",
            "weight_incident_equal_raw",
            "weight_incident_class_equal_raw",
        ]
    ].merge(
        feature_table,
        on="unit_uid",
        how="left",
        validate="one_to_one",
    )

    assert len(
        data
    ) == EXPECTED_UNITS

    del feature_table
    gc.collect()

    X_all = feature_frame_to_float32(
        data,
        features,
    )

    y_all = data[
        "hold_label"
    ].to_numpy(
        dtype=np.int8
    )

    for experiment in experiments_here:

        experiment_id = experiment[
            "experiment_id"
        ]

        weight_scheme = experiment[
            "weight_scheme"
        ]

        print()
        print("-" * 110)
        print(
            "EXPERIMENT:",
            experiment_id
        )
        print("-" * 110)

        fold_prediction_frames = []

        for outer_fold in range(
            OUTER_FOLDS
        ):

            train_mask = (
                data[
                    "outer_fold"
                ].to_numpy()
                != outer_fold
            )

            test_mask = ~train_mask

            train_idx = np.flatnonzero(
                train_mask
            )

            test_idx = np.flatnonzero(
                test_mask
            )

            train_design = data.iloc[
                train_idx
            ]

            weights = (
                normalized_training_weights(
                    train_design,
                    weight_scheme,
                )
            )

            model = make_model(
                MODEL_SEED
                + outer_fold
            )

            t0 = time.time()

            model.fit(
                X_all.iloc[
                    train_idx
                ],
                y_all[
                    train_idx
                ],
                sample_weight=weights,
            )

            fit_seconds = (
                time.time()
                - t0
            )

            pred = model.predict_proba(
                X_all.iloc[
                    test_idx
                ]
            )[
                :,
                1
            ]

            pred = clip_prob(
                pred
            )

            fold_frame = data.iloc[
                test_idx
            ][
                [
                    "unit_uid",
                    "_irwin_norm",
                    "IncidentName",
                    "Year",
                    "hold_label",
                    "outer_fold",
                ]
            ].copy()

            fold_frame[
                "pred_prob"
            ] = pred

            fold_frame[
                "experiment_id"
            ] = experiment_id

            fold_frame[
                "feature_set"
            ] = feature_set

            fold_frame[
                "weight_scheme"
            ] = weight_scheme

            fold_prediction_frames.append(
                fold_frame
            )

            fold_metrics, _, _ = (
                summarize_predictions(
                    fold_frame
                )
            )

            all_fold_metric_rows.append({
                "experiment_id":
                experiment_id,

                "feature_set":
                feature_set,

                "weight_scheme":
                weight_scheme,

                "outer_fold":
                outer_fold,

                **fold_metrics,
            })

            all_runtime_rows.append({
                "experiment_id":
                experiment_id,

                "feature_set":
                feature_set,

                "weight_scheme":
                weight_scheme,

                "outer_fold":
                outer_fold,

                "train_units":
                len(
                    train_idx
                ),

                "test_units":
                len(
                    test_idx
                ),

                "feature_columns":
                len(
                    features
                ),

                "fit_seconds":
                fit_seconds,
            })

            print(
                f"Fold {outer_fold}: "
                f"fit={fit_seconds/60:.2f} min | "
                f"PR-AUC={fold_metrics['pooled_pr_auc']:.4f} | "
                f"ROC-AUC={fold_metrics['pooled_roc_auc']:.4f} | "
                f"Brier={fold_metrics['pooled_brier']:.4f}"
            )

            del model
            del weights
            gc.collect()

        oof = pd.concat(
            fold_prediction_frames,
            ignore_index=True,
        )

        assert len(
            oof
        ) == EXPECTED_UNITS

        assert oof[
            "unit_uid"
        ].is_unique

        assert set(
            oof[
                "outer_fold"
            ]
        ) == set(
            range(
                OUTER_FOLDS
            )
        )

        prediction_path = (
            PREDICTION_ROOT /
            f"{experiment_id}.parquet"
        )

        oof.to_parquet(
            prediction_path,
            index=False,
        )

        prediction_paths[
            experiment_id
        ] = str(
            prediction_path
        )

        print(
            "Saved:",
            prediction_path
        )

        del oof
        del fold_prediction_frames
        gc.collect()

    del X_all
    del data
    gc.collect()


fold_metrics_df = pd.DataFrame(
    all_fold_metric_rows
)

runtime_df = pd.DataFrame(
    all_runtime_rows
)

fold_metrics_df.to_csv(
    OUTPUT_ROOT /
    "OUTER_FOLD_METRICS_V1.csv",
    index=False,
)

runtime_df.to_csv(
    OUTPUT_ROOT /
    "MODEL_RUNTIME_V1.csv",
    index=False,
)

print()
print(
    "ALL LIGHTGBM OUTER-FOLD FITS COMPLETE:",
    len(
        runtime_df
    ),
)

## 8. Aggregate OOF metrics, incident metrics, and calibration tables

In [ ]:
oof_summary_rows = []
incident_frames = []
reliability_frames = []

# Prior first.
prior_metrics_row = {
    "experiment_id":
    "TRAINING_PREVALENCE_PRIOR",

    "feature_set":
    "NONE",

    "weight_scheme":
    "row",

    **prior_metrics,
}

oof_summary_rows.append(
    prior_metrics_row
)

prior_incident[
    "experiment_id"
] = "TRAINING_PREVALENCE_PRIOR"

incident_frames.append(
    prior_incident
)

prior_rel[
    "experiment_id"
] = "TRAINING_PREVALENCE_PRIOR"

reliability_frames.append(
    prior_rel
)


for experiment in EXPERIMENTS:

    experiment_id = experiment[
        "experiment_id"
    ]

    path = Path(
        prediction_paths[
            experiment_id
        ]
    )

    oof = pd.read_parquet(
        path
    )

    metrics, inc, rel = (
        summarize_predictions(
            oof
        )
    )

    oof_summary_rows.append({
        "experiment_id":
        experiment_id,

        "feature_set":
        experiment[
            "feature_set"
        ],

        "weight_scheme":
        experiment[
            "weight_scheme"
        ],

        **metrics,
    })

    inc[
        "experiment_id"
    ] = experiment_id

    incident_frames.append(
        inc
    )

    rel[
        "experiment_id"
    ] = experiment_id

    reliability_frames.append(
        rel
    )


oof_summary = pd.DataFrame(
    oof_summary_rows
)

per_incident_metrics = pd.concat(
    incident_frames,
    ignore_index=True,
)

reliability_bins = pd.concat(
    reliability_frames,
    ignore_index=True,
)

display(
    oof_summary.sort_values(
        "pooled_pr_auc",
        ascending=False,
    )
)

oof_summary.to_csv(
    OUTPUT_ROOT /
    "OOF_METRIC_SUMMARY_V1.csv",
    index=False,
)

per_incident_metrics.to_parquet(
    OUTPUT_ROOT /
    "PER_INCIDENT_METRICS_V1.parquet",
    index=False,
)

reliability_bins.to_csv(
    OUTPUT_ROOT /
    "RELIABILITY_BINS_V1.csv",
    index=False,
)

## 9. Fold variability summary

The fold-2 class distribution is intentionally more Burned-Over-heavy than the
other folds. We therefore report fold spread rather than hiding it through a
row-random split.

In [ ]:
fold_metric_columns = [
    "pooled_pr_auc",
    "pooled_roc_auc",
    "pooled_brier",
    "pooled_log_loss",
    "incident_macro_brier",
    "incident_macro_log_loss",
    "mixed_incident_macro_pr_auc",
    "mixed_incident_macro_roc_auc",
    "calibration_intercept",
    "calibration_slope",
    "ece",
]

fold_summary_rows = []

for experiment_id, g in fold_metrics_df.groupby(
    "experiment_id",
    sort=False,
):

    row = {
        "experiment_id":
        experiment_id,

        "feature_set":
        g[
            "feature_set"
        ].iloc[
            0
        ],

        "weight_scheme":
        g[
            "weight_scheme"
        ].iloc[
            0
        ],
    }

    for metric in fold_metric_columns:

        row[
            f"{metric}_mean"
        ] = float(
            g[
                metric
            ].mean()
        )

        row[
            f"{metric}_sd"
        ] = float(
            g[
                metric
            ].std(
                ddof=1
            )
        )

        row[
            f"{metric}_min"
        ] = float(
            g[
                metric
            ].min()
        )

        row[
            f"{metric}_max"
        ] = float(
            g[
                metric
            ].max()
        )

    fold_summary_rows.append(
        row
    )


fold_variability = pd.DataFrame(
    fold_summary_rows
)

display(
    fold_variability[
        [
            "experiment_id",
            "pooled_pr_auc_mean",
            "pooled_pr_auc_sd",
            "pooled_roc_auc_mean",
            "pooled_roc_auc_sd",
            "pooled_brier_mean",
            "pooled_brier_sd",
            "mixed_incident_macro_pr_auc_mean",
            "mixed_incident_macro_pr_auc_sd",
        ]
    ].sort_values(
        "pooled_pr_auc_mean",
        ascending=False,
    )
)

fold_variability.to_csv(
    OUTPUT_ROOT /
    "OUTER_FOLD_METRIC_VARIABILITY_V1.csv",
    index=False,
)

## 10. Feature-ablation and weighting-ablation tables

These are the two primary 05C comparisons.

### Feature ablation

Every model uses incident-equal training weight.

### Weight ablation

Every model uses `E_FULL_EXTENDED`.

In [ ]:
feature_ablation_ids = [
    e[
        "experiment_id"
    ]
    for e in EXPERIMENTS
    if e[
        "weight_scheme"
    ] == "incident_equal"
]

feature_ablation = (
    oof_summary[
        oof_summary[
            "experiment_id"
        ].isin(
            feature_ablation_ids
        )
    ]
    .copy()
    .sort_values(
        "pooled_pr_auc",
        ascending=False,
    )
)

weight_ablation_ids = [
    "E_FULL_EXTENDED__row",
    "E_FULL_EXTENDED__incident_equal",
    "E_FULL_EXTENDED__incident_class_equal",
]

weight_ablation = (
    oof_summary[
        oof_summary[
            "experiment_id"
        ].isin(
            weight_ablation_ids
        )
    ]
    .copy()
    .sort_values(
        "pooled_pr_auc",
        ascending=False,
    )
)

display(
    feature_ablation[
        [
            "experiment_id",
            "pooled_pr_auc",
            "pooled_roc_auc",
            "pooled_brier",
            "incident_macro_brier",
            "mixed_incident_macro_pr_auc",
            "mixed_incident_macro_roc_auc",
            "calibration_intercept",
            "calibration_slope",
            "ece",
        ]
    ]
)

display(
    weight_ablation[
        [
            "experiment_id",
            "pooled_pr_auc",
            "pooled_roc_auc",
            "pooled_brier",
            "incident_macro_brier",
            "mixed_incident_macro_pr_auc",
            "mixed_incident_macro_roc_auc",
            "calibration_intercept",
            "calibration_slope",
            "ece",
        ]
    ]
)

feature_ablation.to_csv(
    OUTPUT_ROOT /
    "FEATURE_ABLATION_OOF_METRICS_V1.csv",
    index=False,
)

weight_ablation.to_csv(
    OUTPUT_ROOT /
    "WEIGHT_ABLATION_OOF_METRICS_V1.csv",
    index=False,
)

## 11. Improvement relative to no-skill prior

For Brier and log loss, **negative change is better**.

For PR-AUC and ROC-AUC, **positive change is better**.

In [ ]:
prior_row = oof_summary[
    oof_summary[
        "experiment_id"
    ] == "TRAINING_PREVALENCE_PRIOR"
].iloc[
    0
]

comparison = oof_summary[
    oof_summary[
        "experiment_id"
    ] != "TRAINING_PREVALENCE_PRIOR"
].copy()

comparison[
    "delta_pr_auc_vs_prior"
] = (
    comparison[
        "pooled_pr_auc"
    ]
    - prior_row[
        "pooled_pr_auc"
    ]
)

comparison[
    "delta_roc_auc_vs_prior"
] = (
    comparison[
        "pooled_roc_auc"
    ]
    - prior_row[
        "pooled_roc_auc"
    ]
)

comparison[
    "delta_brier_vs_prior"
] = (
    comparison[
        "pooled_brier"
    ]
    - prior_row[
        "pooled_brier"
    ]
)

comparison[
    "delta_log_loss_vs_prior"
] = (
    comparison[
        "pooled_log_loss"
    ]
    - prior_row[
        "pooled_log_loss"
    ]
)

display(
    comparison[
        [
            "experiment_id",
            "delta_pr_auc_vs_prior",
            "delta_roc_auc_vs_prior",
            "delta_brier_vs_prior",
            "delta_log_loss_vs_prior",
        ]
    ].sort_values(
        "delta_pr_auc_vs_prior",
        ascending=False,
    )
)

comparison.to_csv(
    OUTPUT_ROOT /
    "IMPROVEMENT_VS_PRIOR_V1.csv",
    index=False,
)

## 12. Final baseline integrity checks

In [ ]:
prediction_audit_rows = []

for experiment in EXPERIMENTS:

    experiment_id = experiment[
        "experiment_id"
    ]

    path = Path(
        prediction_paths[
            experiment_id
        ]
    )

    pred = pd.read_parquet(
        path,
        columns=[
            "unit_uid",
            "_irwin_norm",
            "hold_label",
            "outer_fold",
            "pred_prob",
        ],
    )

    group_fold_count = (
        pred.groupby(
            "_irwin_norm"
        )[
            "outer_fold"
        ].nunique()
    )

    prediction_audit_rows.append({
        "experiment_id":
        experiment_id,

        "rows":
        len(
            pred
        ),

        "unique_units":
        pred[
            "unit_uid"
        ].nunique(),

        "unique_incidents":
        pred[
            "_irwin_norm"
        ].nunique(),

        "all_probabilities_finite":
        bool(
            np.isfinite(
                pred[
                    "pred_prob"
                ].to_numpy(
                    float
                )
            ).all()
        ),

        "all_probabilities_strictly_between_zero_one":
        bool(
            (
                (
                    pred[
                        "pred_prob"
                    ] > 0
                )
                & (
                    pred[
                        "pred_prob"
                    ] < 1
                )
            ).all()
        ),

        "each_incident_one_outer_fold":
        bool(
            (
                group_fold_count
                == 1
            ).all()
        ),
    })


prediction_audit = pd.DataFrame(
    prediction_audit_rows
)

display(
    prediction_audit
)

checks = [
    {
        "check":
        "exact_eight_lightgbm_experiments",

        "pass":
        len(
            prediction_audit
        ) == 8,
    },
    {
        "check":
        "all_prediction_tables_304143_rows",

        "pass":
        bool(
            (
                prediction_audit[
                    "rows"
                ] == EXPECTED_UNITS
            ).all()
        ),
    },
    {
        "check":
        "all_prediction_tables_unique_units",

        "pass":
        bool(
            (
                prediction_audit[
                    "unique_units"
                ] == EXPECTED_UNITS
            ).all()
        ),
    },
    {
        "check":
        "all_prediction_tables_381_incidents",

        "pass":
        bool(
            (
                prediction_audit[
                    "unique_incidents"
                ] == EXPECTED_INCIDENTS
            ).all()
        ),
    },
    {
        "check":
        "all_predictions_finite",

        "pass":
        bool(
            prediction_audit[
                "all_probabilities_finite"
            ].all()
        ),
    },
    {
        "check":
        "all_predictions_in_open_unit_interval",

        "pass":
        bool(
            prediction_audit[
                "all_probabilities_strictly_between_zero_one"
            ].all()
        ),
    },
    {
        "check":
        "each_incident_entirely_one_outer_fold",

        "pass":
        bool(
            prediction_audit[
                "each_incident_one_outer_fold"
            ].all()
        ),
    },
    {
        "check":
        "fixed_hyperparameters_no_outer_tuning",

        "pass":
        True,
    },
    {
        "check":
        "no_threshold_selected",

        "pass":
        True,
    },
    {
        "check":
        "test_metrics_unweighted",

        "pass":
        True,
    },
    {
        "check":
        "training_weight_mean_normalized_only",

        "pass":
        True,
    },
]

checks_df = pd.DataFrame(
    checks
)

display(
    checks_df
)

BASELINE_PASS = bool(
    checks_df[
        "pass"
    ].all()
)

print()
print(
    "05C INCIDENT-HELD-OUT BASELINE MODELING:",
    (
        "PASS / READY FOR BASELINE SCIENTIFIC INTERPRETATION"
        if BASELINE_PASS
        else "REVIEW REQUIRED"
    ),
)

prediction_audit.to_csv(
    OUTPUT_ROOT /
    "PREDICTION_INTEGRITY_AUDIT_V1.csv",
    index=False,
)

checks_df.to_csv(
    OUTPUT_ROOT /
    "FINAL_BASELINE_CHECKS_V1.csv",
    index=False,
)

## 13. Final report

In [ ]:
def tomd(
    df,
):
    try:
        return df.to_markdown(
            index=False
        )
    except Exception:
        return df.to_string(
            index=False
        )


report = f"""# 05C Incident-Held-Out Probabilistic Baseline Modeling

Build: `{NOTEBOOK_BUILD_ID}`

Generated UTC: {datetime.now(timezone.utc).isoformat()}

## Model

Family: **{MODEL_FAMILY}**

Fixed baseline parameters:

```json
{json.dumps(LGBM_PARAMS, indent=2)}
```

No hyperparameter tuning was performed.

## Experiment matrix

{tomd(experiment_table)}

## OOF metric summary

{tomd(oof_summary.sort_values('pooled_pr_auc', ascending=False))}

## Feature ablation

{tomd(feature_ablation)}

## Weight ablation

{tomd(weight_ablation)}

## Improvement relative to training-prevalence prior

{tomd(comparison)}

## Outer-fold variability

{tomd(fold_variability)}

## Runtime

{tomd(runtime_df.groupby(['experiment_id'], as_index=False).agg(
    fits=('outer_fold', 'size'),
    total_fit_min=('fit_seconds', lambda x: x.sum()/60.0),
    mean_fit_min=('fit_seconds', lambda x: x.mean()/60.0),
))}

## Prediction integrity

{tomd(prediction_audit)}

## Final checks

{tomd(checks_df)}

## Decision

`{"PASS / READY FOR BASELINE SCIENTIFIC INTERPRETATION" if BASELINE_PASS else "REVIEW REQUIRED"}`

05C answers baseline learnability using unseen-fire outer-fold predictions. It
does not yet claim an optimized final model.
"""

REPORT_PATH = (
    OUTPUT_ROOT /
    "INCIDENT_HELD_OUT_BASELINE_MODELING_REPORT_V1.md"
)

REPORT_PATH.write_text(
    report,
    encoding="utf-8",
)

print(
    report
)

print()
print(
    "Saved:",
    REPORT_PATH,
)

## 14. Output inventory

In [ ]:
print("=" * 100)
print("05C OUTPUTS")
print("=" * 100)

for p in sorted(
    OUTPUT_ROOT.glob(
        "*"
    )
):
    if p.is_file():

        print(
            f"{p.name:<95}"
            f"{p.stat().st_size/(1024**2):>12.3f} MB"
        )

print()

for p in sorted(
    PREDICTION_ROOT.glob(
        "*.parquet"
    )
):

    print(
        f"{p.name:<95}"
        f"{p.stat().st_size/(1024**2):>12.3f} MB"
    )

print()
print(
    "DONE:",
    NOTEBOOK_BUILD_ID,
)